
# <font color="green">2-3 Tree</font>

* Let's consider how to implement a 2-3 tree, a simplified version of a B-tree commonly used for indexing data stored in databases.

## 2-3 Tree Definition

* A 2-3 tree is a tree structure similar to the binary search tree discussed earlier (`recursive_type/bstree`).
* See the [Wikipedia article](https://en.wikipedia.org/wiki/2%E2%80%933_tree) for an illustration.
* In a nutshell:
  - Only leaf nodes hold values.
  - Each internal node has either 2 or 3 children, and all leaves are at the same depth (except in the empty and singleton cases); in other words, all children of a node always have the same height.
  - When a node has children $u_0$, $u_1$, and possibly $u_2$, the following holds: any value in $u_0 \le$ any value in $u_1 \le$ any value in $u_2$. This is a property analogous to that of a binary search tree.
  - Internal nodes maintain value(s) that separate their children. Specifically, a node with two children has a separator value $s_0$ such that any value in $u_0 < s_0 \le$ any value in $u_1$; a node with three children has two separator values $s_0$ and $s_1$ such that any value in $u_0 < s_0 \le$ any value in $u_1 < s_1 \le$ any value in $u_2$.
  - Due to this ordering property and the balance property mentioned above, the tree guarantees $O(\log n)$ performance for search.
* The main question is how to insert a value into, or remove a value from, a tree while maintaining the equal-height property in $O(\log n)$ time.
* Below, we refer to a node with two children as a 2-node and a node with three children as a 3-node.

## In This Problem ...

* Define a data type representing a 2-3 tree, called `tttree` (or `TTTree`)
* Define four functions explained below `tttree_add, tttree_lookup, tttree_remove,` and `tttree_check` (or `TTTreeAdd, TTTreeLookup, TTTreeRemove`, and `TTTreeCheck`)
* There are implicit conditions imposed by test code; specifically
  * In Go, use `nil` to represent an empty tree
  * In Julia, use `nothing` to represent an empty tree
  * In OCaml, use variant to define 2-3 tree and `Empty` to represent an empty tree
  * In Rust, use `enum` to define 2-3 tree and `Empty` (`TTTree::Empty`) to represent an empty tree
* Check the test code before you proceed


## Adding a Value

Let's consider adding an element $x$ to a 2-3 tree $t$.
* (a) If $t$ is empty, create a singleton tree containing $x$.
* (b) If $t$ is a singleton tree containing $y$, create a 2-node with two leaf children $x$ and $y$.
* (c) Otherwise $t$ should have a 2- or 3-node as its root; choose the appropriate child $u$ by comparing $x$ and the separator value(s), then add $x$ to $u$.

However, simply replacing $u$ with the result of adding $x$ to $u$ may break the equal-height property. Let's consider how to handle this, starting with the simple case of depth 2 (i.e., a 2- or 3-node whose children are leaves).
* (*c2) If $t$ is a 2-node (i.e., has two leaf children $x_0$ and $x_1$), turn it into a 3-node with three leaves $x_0$, $x_1$, and $x$ in the appropriate order.
* (*c3) If $t$ is a 3-node (i.e., has three leaf children $x_0$, $x_1$, and $x_2$), simply adding $x$ would produce a 4-node, which is not allowed; we therefore split it into two 2-nodes.

With this in mind, let's consider the general case where a subtree may be split during insertion.
* (c) As before, first choose the appropriate child $u$ and add $x$ to it.
* (c-i) If $u$ is not split, simply replace $u$ with the result of adding $x$ to $u$.
* (c-ii) If $u$ is split:
  * If $t$ is a 2-node, create a 3-node by replacing $u$ with the two resulting nodes.
  * If $t$ is a 3-node, split it into two 2-nodes; this split is then handled by the parent.

* Define a function `tttree_add` that takes a non-negative integer $x$ and a 2-3 tree $t$, and inserts $x$ into $t$.
* You may assume $t$ does not already contain $x$ (no duplicates).
* It should not update $t$ in place; instead, it returns a new tree containing $x$.

## Removing a Value

Now let's consider removing a value $x$ from $t$.
* (a) If $t$ is empty, it remains empty.
* (b) If $t$ is a singleton, it becomes empty if $x$ matches its value, or remains unchanged otherwise.
* (c) Otherwise $t$'s root should be a 2- or 3-node; choose the appropriate child $u$ and remove $x$ from $u$.

Again, simply replacing $u$ with the result of removing $x$ from $u$ may break the equal-height property. As before, let's start with the simple case of depth 2 (i.e., a 2- or 3-node whose children are leaves).
* (*c3) If $t$ is a 3-node (i.e., has three leaf children $x_0$, $x_1$, and $x_2$) and one of the three values is removed, turn it into a 2-node.
* (*c2) If $t$ is a 2-node (i.e., has two leaf children $x_0$ and $x_1$) and one value is removed, the result would be a 1-node, which is not allowed; return it with a special flag indicating it is an invalid 1-node.

With this in mind, let's consider the general case where a 2-node may become a 1-node during removal.
* (c) As before, first choose the appropriate child $u$ and remove $x$ from it.
  * (i) If $u$ does not become a 1-node, simply replace $u$ with the result of removing $x$ from $u$.
  * (ii) If $u$ becomes a 1-node:
    * If $t$ is a 3-node, merge the 1-node with a neighboring sibling (if the 1-node is the middle child, either sibling may be used). If the sibling is a 3-node, split it into two 2-nodes ($t$ remains a 3-node); if the sibling is a 2-node, merge them into a 3-node ($t$ becomes a 2-node).
    * If $t$ is a 2-node, merge the 1-node with the other sibling. If the sibling is a 3-node, split it into two 2-nodes ($t$ remains a 2-node); if the sibling is a 2-node, merge them into a 3-node, and $t$ becomes a 1-node, which is then handled by the parent.

* Define a function `tttree_remove` that takes a non-negative integer $x$ and a 2-3 tree $t$, and removes $x$ from $t$ if it is present.
* You may assume each value appears at most once in $t$.
* It should not update $t$ in place; instead, it returns a new tree without $x$.
* If $x$ is not in $t$, return $t$ unchanged.

## Looking Up a Value

* Looking up a value is straightforward.
* Define a function `tttree_lookup` that takes a non-negative integer $x$ and a 2-3 tree $t$, and checks whether $x$ is in $t$.
* It should return `true` if $x$ is in $t$, and `false` otherwise.

## Checking a Tree

* Define a function `tttree_check` that takes a 2-3 tree $t$ and returns its depth, while verifying that all children of every internal node have the same height.
  * The depth of an empty tree is 0.
  * The depth of a singleton tree or a tree consisting of a root + 2 or 3 leaves is 1.
  * The depth of a 2- or 3-node is the depth of its children (which must all be equal) plus 1.

## Note

* The generalization of a 2-3 tree is the B-tree, widely used for indexing in databases.
* Each internal node of a B-tree has between $m$ and $2m-1$ children; a 2-3 tree corresponds to the case $m = 2$.

## Boilerplate and Test Code

* Boilerplate source files `{go,jl,ml,rs}/tttree.{go,jl,ml,rs}` containing the test code are generated and shown below.
* The test code does the following:
   - Randomly shuffles $0, 1, 2, \ldots, n-1$; call the result $X$.
   - Inserts all elements of $X$ into an empty tree (`tttree_add`).
   - Checks that each element of $X$ is found in the tree (`tttree_lookup`).
   - Randomly shuffles $X$ again; call the result $Y$.
   - Removes all elements of $Y$ from the resulting tree (`tttree_remove`).
   - Checks that the resulting tree is empty.
* Edit the source files either by opening them in a text editor (e.g., VSCode), or by editing the cells below and executing them.





# 1. AI tutor
## 1-1. Prepare
* Your personal AI tutor is provided for questions and feedback
* Execute the following cell before you use it

In [ ]:
import heytutor

## 1-2. Examples
### 1-2-1. A general question
```
%%hey
How to write a function in Go?
```

### 1-2-2. A hint on this specific problem
```
%%hey
Give me a hint on this problem for Rust
```

### 1-2-3. <font color="red">NEW:</font> A few builtin variables
* `{file:FILENAME}` is the content of FILE
* `{bash[-1]}` is the output of the last `%%bash_` cell, `{bash[-2]}` that of the second last `%%bash_` cell, etc.
* `{problem}` is the content of the file you specified by `%%hey problem_file=foo.md`
* `{answer}` is the content of the file you specified by `%%hey answer_file=go/foo.go`

### 1-2-4. Help when you struggle
```
%%hey answer_file=go/foo.go
I get this error when I compile it. What's wrong?"

My program:
{answer}

Error message:
{bash[-1]}

```

### 1-2-5. Ask feedback
* You are encouraged to ask a feedback once you think you are done with the problem, to know if there is a better answer.  You can do so by something like:

```
%%hey problem_file=foo.md answer_file=go/foo.md
Give me a feedback to my answer.

Problem:
{problem}

My Answer:
{answer}
```

# 2. Go

## 2-1. Baseline code

In [ ]:
import heytutor

In [ ]:
%%writefile_ go/tttree.go
package main
import (
	"fmt"
	"slices"
	"strings"
)

/** begin my answer */

/** end my answer */

// add all values in list xs to tree t in order
func addSeq(xs []int, t *TTTree) *TTTree {
	for _, x := range xs {
		t = tttreeAdd(x, t)
	}
	return t
}

func lookupSeq(xs []int, t *TTTree) bool {
	for _, x := range xs {
		if !tttreeLookup(x, t) {
			return false
		}
	}
	return true
}

func removeSeq(xs []int, t *TTTree) *TTTree {
	for _, x := range xs {
		t = tttreeRemove(x, t)
	}
	return t
}

// 0 ... n - 1
func mkRange(n int) []int {
	s := make([]int, n)
	for i := 0; i < n; i++ {
		s[i] = i
	}
	return s
}

// random sequence
func randomSeq(n int, x int) []int {
	seq := make([]int, n)
	for i := 0; i < n; i++ {
		x = (0x5DEECE66D * x + 0xB) & 0xFFFFFFFFFFFF
		seq[i] = x >> 17
	}
	return seq
}

// randomly shuffle xs
type Pair struct {
	k int
	x int
}

func printSlice(header string, xs []int, n int) {
	fmt.Print(header, "[")
	for i, x := range xs {
		if i >= n {
			fmt.Print(" ...")
			break
		}
		if i > 0 {
			fmt.Print(", ")
		}
		fmt.Print(x)
	}
	fmt.Println("]")
}

func randomShuffle(seed int, xs []int) []int {
	ys := randomSeq(len(xs), seed)
	ps := make([]Pair, len(xs))
	for i, x := range xs {
		ps[i] = Pair{k: ys[i], x: x}
	}
	slices.SortFunc(ps, func(a, b Pair) int {
		return a.k - b.k
	})
	for i, z := range ps {
		ys[i] = z.x
	}
	return ys
}

func checkSeq(xs0 []int, xs1 []int) bool {
	printSlice("insert in this order: ", xs0, 7)
	printSlice("remove in this order: ", xs1, 7)
	t := addSeq(xs0, nil)
	tttreeCheck(t)
	if lookupSeq(xs1, t) {
		e := removeSeq(xs1, t)
		return e == nil
	} else {
		return false
	}
}

// randomly shuffle 0 .. n-1;
//   add them to empty tree;
//   remove all elements from the tree;
//   check they are sorted

func checkRandom(insertSeed int, removeSeed, n int) bool {
	rng := mkRange(n)
	xs0 := randomShuffle(insertSeed, rng)
	xs1 := randomShuffle(removeSeed, rng)
	return checkSeq(xs0, xs1)
}

func main() {
	insertSeed := 1234
	removeSeed := 123456
	if true {
		if !checkRandom(insertSeed, removeSeed, 2) { panic("wrong") }
		if !checkRandom(insertSeed, removeSeed, 3) { panic("wrong") }
		if !checkRandom(insertSeed + 0, removeSeed + 0, 4) { panic("wrong") }
		if !checkRandom(insertSeed + 1, removeSeed + 1, 4) { panic("wrong") }
		if !checkRandom(insertSeed + 2, removeSeed + 2, 4) { panic("wrong") }
		if !checkRandom(insertSeed + 3, removeSeed + 3, 4) { panic("wrong") }
		if !checkRandom(insertSeed + 0, removeSeed + 0, 5) { panic("wrong") }
		if !checkRandom(insertSeed + 1, removeSeed + 1, 5) { panic("wrong") }
		if !checkRandom(insertSeed + 2, removeSeed + 2, 5) { panic("wrong") }
		if !checkRandom(insertSeed + 3, removeSeed + 3, 5) { panic("wrong") }
		if !checkRandom(insertSeed, removeSeed, 10) { panic("wrong") }
	}
	if !checkRandom(insertSeed, removeSeed, 12) { panic("wrong") }
	if true {
		if !checkRandom(insertSeed, removeSeed, 1000) { panic("wrong") }
		if !checkRandom(insertSeed, removeSeed, 10000) { panic("wrong") }
		if !checkRandom(insertSeed, removeSeed, 100000) { panic("wrong") }
	}
	fmt.Println("OK")
}

## 2-2. Compile

In [ ]:
%%bash_
export PATH=${PATH}:~/.local/go/bin:~/go/bin
go build -o go/tttree go/tttree.go

* Note: when you run `go` or other Go commands in a terminal (SSH or Jupyter terminal), you need to execute the first line (`export PATH=${PATH}:~/go/bin`)
* You may consider adding that line in your `~/.bash_profile`

## 2-3. Run

In [ ]:
%%bash_
go/tttree

## 2-4. Ask Questions or Get Feedback

In [ ]:
%%hey problem_file=tttree.md answer_file=go/tttree.go

Problem:
{problem}
My Answer (between /** begin my answer */ and /** end my answer */):
{answer}

Give me a feedback to my answer.

# 3. Julia

## 3-1. Baseline code

In [ ]:
import heytutor

In [ ]:
%%writefile_ jl/tttree.jl
# begin my answer

# begin hidden
# non-empty two-three-tree
# leaf : length(keys) == 1, children == nil
# N2 : length(keys) == 1, length(children) == 2
# N3 : length(keys) == 2, length(children) == 3
struct TTTree_
    keys
    children
end

function leaf(x)
    TTTree_(Int[x], nothing)
end

function n2(u0, s0, u1)
    TTTree_(Int[s0], [u0, u1])
end

function n3(u0, s0, u1, s1, u2)
    TTTree_(Int[s0, s1], [u0, u1, u2])
end

function is_leaf(t)
    t.children === nothing
end

# empty : nil
# singleton : t = nil
# tree : t != nil
struct TTTree
    key
    t
end

function singleton(x)
    TTTree(x, nothing)
end

function tree(t)
    TTTree(-1, t)
end

function is_empty(t)
    t === nothing
end

function is_singleton(t)
    t.t === nothing
end

#= check if all leaves have the same depth and all internal nodes have the correct number of children (2 or 3)
   and returns the depth of the tree if it is a valid two-three-tree =#
function tttree_check_(t)
    if is_leaf(t)       # leaf
        0
    else
        n = length(t.children)
        d = tttree_check_(t.children[1])
        for i in 2:n
            d_ = tttree_check_(t.children[i])
            if d_ != d; error("assert all children have the same depth"); end
        end
        d + 1
    end
end

function _tttree_check(t)
    if is_empty(t)          # empty
        0
    elseif is_singleton(t)  # singleton
        1
    else                    # tree
        tttree_check_(t.t)
    end
end

function tttree_print_(t, depth)
    if is_leaf(t)
        print(repeat(" ", depth), "Leaf(", t.keys[1], ")\n")
        0
    else
        print(repeat(" ", depth), "Node(")
        n = length(t.children)
        for i in 1:n-1
            if i > 1
                print("|")
            end
            print(t.keys[i])
        end
        print(") [\n")
        d = tttree_print_(t.children[1], depth + 1)
        for i in 2:n
            d_ = tttree_print_(t.children[i], depth + 1)
            if d_ != d; error("assert all children have the same depth"); end
        end
        print(repeat(" ", depth), "]\n")
        d + 1
    end
end

function tttree_check(t)
    if is_empty(t)
        #println("Empty")
        0
    elseif is_singleton(t)
        #println("Singleton(", t.key, ")")
        1
    else
        tttree_check_(t.t)
    end
end

# return
# (nil, c0, s0, c1) or
# (t', nil, -1, nil)
function tttree_add_(x, t)
    if is_leaf(t)
        y = t.keys[1]
        u, v = min(x, y), max(x, y)
        nothing, leaf(u), v, leaf(v)
    elseif length(t.children) == 2
        #    s0
        # u0    u1
        s0 = t.keys[1]
        u0, u1 = t.children[1], t.children[2]
        if x < s0
            u0_, u00, s00, u01 = tttree_add_(x, u0)
            if u0_ === nothing
                #     s00     s0
                # u00     u01    u1
                n3(u00, s00, u01, s0, u1), nothing, -1, nothing
            else
                n2(u0_, s0, u1), nothing, -1, nothing
            end
        else
            u1_, u10, s10, u11 = tttree_add_(x, u1)
            if u1_ === nothing
                #    s0     s10
                # u0    u10     u11
                n3(u0, s0, u10, s10, u11), nothing, -1, nothing
            else
                n2(u0, s0, u1_), nothing, -1, nothing
            end
        end
    elseif length(t.children) == 3
        s0, s1 = t.keys[1], t.keys[2]
        u0, u1, u2 = t.children[1], t.children[2], t.children[3]
        if x < s0
            u0_, u00, s00, u01 = tttree_add_(x, u0)
            if u0_ === nothing
                #     s00
                # u00     s0     s1
                #     u01     u1    u2
                nothing, n2(u00, s00, u01), s0, n2(u1, s1, u2)
            else
                n3(u0_, s0, u1, s1, u2), nothing, -1, nothing
            end
        elseif x < s1
            u1_, u10, s10, u11 = tttree_add_(x, u1)
            if u1_ === nothing
                #    s0     s10      s1
                # u0    u10      u11    u2
                nothing, n2(u0, s0, u10), s10, n2(u11, s1, u2)
            else
                n3(u0, s0, u1_, s1, u2), nothing, -1, nothing
            end
        else
            u2_, u20, s20, u21 = tttree_add_(x, u2)
            if u2_ === nothing
                #    s0    s1     s21
                # u0    u1    u20     u21
                nothing, n2(u0, s0, u1), s1, n2(u20, s20, u21)
            else
                n3(u0, s0, u1, s1, u2_), nothing, -1, nothing
            end
        end
    else
        error("assert invalid node")
    end
end

function tttree_add(x, t)
    if is_empty(t)
        singleton(x)
    elseif is_singleton(t)
        y = t.key
        u, v = min(x, y), max(x, y)
        tree(n2(leaf(u), v, leaf(v)))
    else
        t_, t0, s0, t1 = tttree_add_(x, t.t)
        if t_ === nothing
            tree(n2(t0, s0, t1))
        else
            tree(t_)
        end
    end
end

function tttree_lookup_(x, t)
    if is_leaf(t)
        x == t.keys[1]
    else
        n = length(t.children)
        for i in 1:n-1
            if x < t.keys[i]
                return tttree_lookup_(x, t.children[i])
            end
        end
        tttree_lookup_(x, t.children[n])
    end
end

function tttree_lookup(x, t)
    if is_empty(t)
        false
    elseif is_singleton(t)
        x == t.key
    else
        tttree_lookup_(x, t.t)
    end
end

# return (nil, nil) if t becomes empty after removing x
# return (nil, c0) if t becomes a single-child node whose sole child is c0
# return (t', nil) if t remains a valid node having 2 or 3 children after removing x
function tttree_remove_(x, t)
    if is_leaf(t)
        if x == t.keys[1]
            nothing, nothing # empty
        else
            t, nothing # unchanged
        end
    elseif length(t.children) == 2
        #    s0
        # u0    u1
        s0 = t.keys[1]
        u0, u1 = t.children[1], t.children[2]
        if x < s0
            u0_, u00 = tttree_remove_(x, u0)
            if u0_ === nothing && u00 === nothing
                # t becomes single-child node
                nothing, u1
            elseif u0_ === nothing
                #      s0
                #  u0_     u1
                #   |
                #  u00
                if length(u1.children) == 2
                    #      s0
                    #  u0_    u1
                    #   |     / \
                    #  u00  u10 u11
                    s10 = u1.keys[1]
                    u10, u11 = u1.children[1], u1.children[2]
                    nothing, n3(u00, s0, u10, s10, u11)
                elseif length(u1.children) == 3
                    #      s0
                    #  u0_    u1
                    #   |     /|\
                    #  u00  u10u11u12
                    s10, s11 = u1.keys[1], u1.keys[2]
                    u10, u11, u12 = u1.children[1], u1.children[2], u1.children[3]
                    n2(n2(u00, s0, u10), s10, n2(u11, s11, u12)), nothing
                else
                    error("assert invalid node")
                end
            else
                n2(u0_, s0, u1), nothing
            end
        else
            u1_, u10 = tttree_remove_(x, u1)
            if u1_ === nothing && u10 === nothing
                nothing, u0
            elseif u1_ === nothing
                #    s0
                # u0    u1_
                #        |
                #       u10
                if length(u0.children) == 2
                    #    s0
                    # u0    u1_
                    # /\     |
                    #u00u01  u10
                    s00 = u0.keys[1]
                    u00, u01 = u0.children[1], u0.children[2]
                    nothing, n3(u00, s00, u01, s0, u10)
                elseif length(u0.children) == 3
                    #      s0
                    #   u0     u1_
                    #  /|\     |
                    #u00u01u02 u10
                    s00, s01 = u0.keys[1], u0.keys[2]
                    u00, u01, u02 = u0.children[1], u0.children[2], u0.children[3]
                    n2(n2(u00, s00, u01), s01, n2(u02, s0, u10)), nothing
                else
                    error("assert invalid node")
                end
            else
                n2(u0, s0, u1_), nothing
            end
        end
    elseif length(t.children) == 3
        #    s0    s1
        # u0    u1    u2
        s0, s1 = t.keys[1], t.keys[2]
        u0, u1, u2 = t.children[1], t.children[2], t.children[3]
        if x < s0
            u0_, u00 = tttree_remove_(x, u0)
            if u0_ === nothing && u00 === nothing
                n2(u1, s1, u2), nothing
            elseif u0_ === nothing
                #    s0    s1
                # u0    u1    u2
                #  |
                # u00
                if length(u1.children) == 2
                    #    s0     s1
                    # u0     u1     u2
                    #  |     / \
                    # u00   u10 u11
                    s10 = u1.keys[1]
                    u10, u11 = u1.children[1], u1.children[2]
                    n2(n3(u00, s0, u10, s10, u11), s1, u2), nothing
                elseif length(u1.children) == 3
                    #    s0     s1
                    # u0     u1     u2
                    #  |     /|\
                    # u00  u10u11u12
                    s10, s11 = u1.keys[1], u1.keys[2]
                    u10, u11, u12 = u1.children[1], u1.children[2], u1.children[3]
                    n3(n2(u00, s0, u10), s10, n2(u11, s11, u12), s1, u2), nothing
                else
                    error("assert invalid node")
                end
            else
                n3(u0_, s0, u1, s1, u2), nothing
            end
        elseif x < s1
            u1_, u10 = tttree_remove_(x, u1)
            if u1_ === nothing && u10 === nothing
                n2(u0, s0, u2), nothing
            elseif u1_ === nothing
                #     s0    s1
                # u0    u1    u2
                #       |
                #      u10
                if length(u0.children) == 2
                    #     s0    s1
                    # u0    u1    u2
                    # /\     |
                    #u00u01  u10
                    s00 = u0.keys[1]
                    u00, u01 = u0.children[1], u0.children[2]
                    n2(n3(u00, s00, u01, s0, u10), s1, u2), nothing
                elseif length(u0.children) == 3
                    #      s0     s1
                    #   u0     u1    u2
                    #  /|\     |
                    #u00u01u02 u10
                    s00, s01 = u0.keys[1], u0.keys[2]
                    u00, u01, u02 = u0.children[1], u0.children[2], u0.children[3]
                    n3(n2(u00, s00, u01), s01, n2(u02, s0, u10), s1, u2), nothing
                else
                    error("assert invalid node")
                end
            else
                n3(u0, s0, u1_, s1, u2), nothing
            end
        else
            u2_, u20 = tttree_remove_(x, u2)
            if u2_ === nothing && u20 === nothing
                n2(u0, s0, u1), nothing
            elseif u2_ === nothing
                #     s0    s1
                # u0    u1    u2
                #             |
                #            u20
                if length(u1.children) == 2
                    #     s0    s1
                    # u0    u1    u2
                    #       /\     |
                    #     u10u11  u20
                    s10 = u1.keys[1]
                    u10, u11 = u1.children[1], u1.children[2]
                    n2(u0, s0, n3(u10, s10, u11, s1, u20)), nothing
                elseif length(u1.children) == 3
                    #      s0     s1
                    #   u0     u1     u2
                    #         /|\     |
                    #      u10u11u12  u20
                    s10, s11 = u1.keys[1], u1.keys[2]
                    u10, u11, u12 = u1.children[1], u1.children[2], u1.children[3]
                    n3(u0, s0, n2(u10, s10, u11), s11, n2(u12, s1, u20)), nothing
                else
                    error("assert invalid node")
                end
            else
                n3(u0, s0, u1, s1, u2_), nothing
            end
        end
    else
        error("assert invalid node")
    end
end

function tttree_remove(x, t)
    if is_empty(t)
        nothing
    elseif is_singleton(t)
        if x == t.key
            nothing
        else
            t
        end
    else
        t_, c0 = tttree_remove_(x, t.t)
        if t_ === nothing && c0 === nothing
            nothing
        elseif t_ === nothing
            if c0.children === nothing
                singleton(c0.keys[1])
            else
                tree(c0)
            end
        else
            tree(t_)
        end
    end
end

# end hidden
# end my answer

# add all values in list xs to tree t in order
function add_seq(xs, t)
    for x in xs
        t = tttree_add(x, t)
    end
    t
end

function lookup_seq(xs, t)
    for x in xs
        if !tttree_lookup(x, t)
            return false
        end
    end
    true
end

function remove_seq(xs, t)
    for x in xs
        t = tttree_remove(x, t)
    end
    t
end

# 0 ... n - 1
function range(n)
    collect(0 : n-1)
end

# random sequence
function random_seq(n, x)
    s = []
    for i in 1:n
        x = (0x5DEECE66D * x + 0xB) & 0xFFFFFFFFFFFF
        push!(s, Int(x >> 17))
    end
    s
end

# randomly shuffle xs
function random_shuffle(seed, xs)
    ys = random_seq(length(xs), seed)
    zs = collect(zip(ys, xs))
    sorted_zs = sort(zs, by=first)
    map(last, sorted_zs)
end

function print_vec(header, xs, n)
    print(header, "[")
    for (i, x) in enumerate(xs)
        if i > n
            print(" ...")
            break
        end
        if i > 1
            print(", ")
        end
        print(x)
    end
    println("]")
end

function check_seq(xs0, xs1)
    print_vec("insert in this order: ", xs0, 7)
    print_vec("remove in this order: ", xs1, 7)
    t = add_seq(xs0, nothing)
    tttree_check(t)
    if lookup_seq(xs1, t)
        e = remove_seq(xs1, t)
        e === nothing
    else
        false
    end
end

# randomly shuffle 0 .. n-1;
#   add them to empty tree;
#   remove all elements from the tree;
#   check they are sorted

function check_random(insert_seed, remove_seed, n)
    rng = range(n)
    xs0 = random_shuffle(insert_seed, rng)
    xs1 = random_shuffle(remove_seed, rng)
    check_seq(xs0, xs1)
end

function main()
    insert_seed = 1234
    remove_seed = 123456
    @assert check_random(insert_seed, remove_seed, 2)
    @assert check_random(insert_seed, remove_seed, 3)
    @assert check_random(insert_seed + 0, remove_seed + 0, 4)
    @assert check_random(insert_seed + 1, remove_seed + 1, 4)
    @assert check_random(insert_seed + 2, remove_seed + 2, 4)
    @assert check_random(insert_seed + 3, remove_seed + 3, 4)
    @assert check_random(insert_seed + 0, remove_seed + 0, 5)
    @assert check_random(insert_seed + 1, remove_seed + 1, 5)
    @assert check_random(insert_seed + 2, remove_seed + 2, 5)
    @assert check_random(insert_seed + 3, remove_seed + 3, 5)
    @assert check_random(insert_seed, remove_seed, 10)
    @assert check_random(insert_seed, remove_seed, 12)
    @assert check_random(insert_seed, remove_seed, 1000)
    @assert check_random(insert_seed, remove_seed, 10000)
    @assert check_random(insert_seed, remove_seed, 100000)
    println("OK")
end

main()

## 3-2. Compile

* Julia code is compiled "just in time" (compiled upon executed), so does not need a specific action for compilation before you run

## 3-3. Run

In [ ]:
%%bash_
export PATH=${PATH}:~/.juliaup/bin
julia jl/tttree.jl

* Note: when you run `julia` or other Julia commands in a terminal (SSH or Jupyter terminal), you need to execute the first line (`export PATH=${PATH}:~/.juliaup/bin`)
* You may consider adding that line in your `~/.bash_profile`

## 3-4. Interactive execution
* `julia` command  also serves is an interactive command for Julia programs

* You can run a source code and continue interaction

```
$ julia -i jl/tttree.jl
```

* For trial and error, you may also consider creating a Julia notebook


## 3-5. Ask Questions or Get Feedback

In [ ]:
%%hey problem_file=tttree.md answer_file=jl/tttree.jl

Problem:
{problem}

My Answer (between ### begin my answer and ### end my answer):
{answer}

Give me a feedback to my answer.

# 4. OCaml

## 4-1. Baseline code

In [ ]:
import heytutor

In [ ]:
%%writefile_ ml/tttree.ml
(** begin my answer *)

(** end my answer *)

let rec add_seq xs t =
  (* let _ = tttree_check t in *)
  match xs with
    [] -> t
  | x :: rs -> add_seq rs (tttree_add x t)
;;

let rec lookup_seq xs t =
  match xs with
    [] -> true
  | x :: rs -> (tttree_lookup x t) && (lookup_seq rs t)
;;

let rec remove_seq xs t =
  (* let _ = tttree_check t in *)
  match xs with
    [] -> t
  | x :: rs -> remove_seq rs (tttree_remove x t)
;;

let rec range n xs =
  if n = 0 then
    xs
  else
    range (n - 1) ((n - 1) :: xs)

(* print int list *)
let rec print_elems xs i n =
  if i = n then
    Printf.printf " ..."
  else
    match xs with
      [] -> ()
    | x :: rs ->
       (if i > 0 then
          Printf.printf ", %d" x
        else
          Printf.printf "%d" x;
        print_elems rs (i + 1) n)
;;

let print_list header xs n =
  Printf.printf "%s[" header;
  print_elems xs 0 n;
  Printf.printf "]\n";
  flush stdout
;;

let rec random_seq_ i n x seq =
  if i = n then
    seq
  else
    let y = (0x5DEECE66D * x + 0xB) land 0xFFFFFFFFFFFF in
    random_seq_ (i + 1) n y ((y lsr 17) :: seq)
;;

let rec random_seq n seed =
  List.rev (random_seq_ 0 n seed [])
;;

let random_shuffle seed xs =
  let rs = random_seq (List.length xs) seed in
  let cs = List.combine rs xs in
  let sorted_cs = List.sort (fun (a, _) (b, _) -> (-) a b) cs in
  let shuffled = List.map snd sorted_cs in
  shuffled
;;

let check_seq xs0 xs1 =
  print_list "insert in this order: " xs0 7;
  print_list "remove in this order: " xs1 7;
  let t = add_seq xs0 Empty in
  let _ = tttree_check t in
  if lookup_seq xs1 t then 
    let e = remove_seq xs1 t in
    e = Empty
  else
    false
;;

let check_random insert_seed remove_seed n =
  let rng = range n [] in
  let xs0 = random_shuffle insert_seed rng in
  let xs1 = random_shuffle remove_seed rng in
  check_seq xs0 xs1
;;

let main () =
  let insert_seed = 1234 in
  let remove_seed = 123456 in
  assert(check_random insert_seed remove_seed 2);
  assert(check_random insert_seed remove_seed 3);
  assert(check_random (insert_seed + 0) (remove_seed + 0) 4);
  assert(check_random (insert_seed + 1) (remove_seed + 1) 4);
  assert(check_random (insert_seed + 2) (remove_seed + 2) 4);
  assert(check_random (insert_seed + 3) (remove_seed + 3) 4);
  assert(check_random (insert_seed + 0) (remove_seed + 0) 5);
  assert(check_random (insert_seed + 1) (remove_seed + 1) 5);
  assert(check_random (insert_seed + 2) (remove_seed + 2) 5);
  assert(check_random (insert_seed + 3) (remove_seed + 3) 5);
  assert(check_random insert_seed remove_seed 10);
  assert(check_random insert_seed remove_seed 100);
  assert(check_random insert_seed remove_seed 1000);
  assert(check_random insert_seed remove_seed 10000);
  assert(check_random insert_seed remove_seed 100000);
  Printf.printf "OK\n"
;;

main()

## 4-2. Compile

In [ ]:
%%bash_
eval $(opam env)
ocamlc ml/tttree.ml -o ml/tttree

* Note: when you run `ocamlc` or other OCaml commands (see below) in a terminal (SSH or Jupyter terminal), you need to execute the first line (`eval $(opam env)`)
* You may consider adding that line in your `~/.bash_profile`

## 4-3. Run

In [ ]:
%%bash_
ml/tttree

## 4-4. Interactive execution
* `ocaml` command is an interactive command for OCaml programs

* In terminal (Jupyter or SSH), you can directly run a source code

```
$ eval $(opam env)   # once in your session or put it in ~/.bash_profile
$ ocaml ml/tttree.ml
```

* You can run a source code and continue interaction

```
$ eval $(opam env)   # once in your session or put it in ~/.bash_profile
$ ocaml -init ml/tttree.ml
```

* For trial and error, you may also consider creating an OCaml notebook


## 4-5. Ask Questions or Get Feedback

In [ ]:
%%hey problem_file=tttree.md answer_file=ml/tttree.ml

Problem:
{problem}

My Answer (between (** begin my answer *) and (** end my answer *)):
{answer}

Give me a feedback to my answer.

# 5. Rust

## 5-1. Baseline code

In [ ]:
import heytutor

In [ ]:
%%writefile_ rs/tttree.rs
/** begin my answer */

/** end my answer */

fn add_seq(xs: &[i64], t: TTTree) -> TTTree {
    let mut t = t;
    for x in xs {
        t = tttree_add(*x, t);
    }
    t
}

fn lookup_seq(xs: &[i64], t: &TTTree) -> bool {
    for x in xs {
        if !tttree_lookup(*x, t) {
            return false;
        }
    }
    true
}

fn remove_seq(xs: &[i64], t: TTTree) -> TTTree {
    let mut t = t;
    for x in xs {
        t = tttree_remove(*x, t);
    }
    t
}

fn range(n : i64) -> Vec<i64> {
    (0..n).collect()
}

fn print_vec(header : &str, v : &[i64], n : usize) {
    print!("{}[", header);
    for (i, x) in v.iter().enumerate() {
	if i >= n {
	    print!("...");
	    break;
	}
	if i > 0 {
	    print!(", ");
	}
	print!("{}", x);
    }
    println!("]");
}

/** random sequence */
fn random_seq(n : i64, mut x : i64) -> Vec<i64> {
    let mut s = vec![];
    for _i in 0..n {
        x = (x.wrapping_mul(0x5DEECE66D) + 0xB) & 0xFFFFFFFFFFFF;
        s.push(x >> 17)
    }
    s
}

/** randomly shuffle xs */
fn random_shuffle(seed : i64, xs : Vec<i64>) -> Vec<i64> {
    let ys = random_seq(xs.len() as i64, seed);
    let mut zs : Vec<(i64,i64)> = ys.iter().copied().zip(xs.iter().copied()).collect();
    zs.sort();
    zs.into_iter().map(|(_k, x)| x).collect()
}

/**
make a tree by inserting elements in xs0;
check if all elements in xs1 are in the tree;
then remove all elements in xs1 from the tree
and check if we are left with empty tree */
fn check_seq(xs0: &[i64], xs1: &[i64]) -> bool {
    print_vec("insert in this order: ", xs0, 7);
    print_vec("remove in this order: ", xs1, 7);
    let t = add_seq(xs0, TTTree::Empty);
    let _ = tttree_check(&t);
    if lookup_seq(xs1, &t) {
        match remove_seq(xs1, t) {
	    TTTree::Empty => true,
	    _ => false,
	}
    } else {
        false
    }
}

fn check_random(insert_seed: i64, remove_seed: i64, n: i64) -> bool {
    let rng = range(n);
    let xs0 = random_shuffle(insert_seed, rng.clone());
    let xs1 = random_shuffle(remove_seed, rng);
    check_seq(&xs0, &xs1)
}

fn main() {
    let insert_seed: i64 = 1234;
    let remove_seed: i64 = 123456;
    assert!(check_random(insert_seed, remove_seed, 2));
    assert!(check_random(insert_seed, remove_seed, 3));
    assert!(check_random(insert_seed + 0, remove_seed + 0, 4));
    assert!(check_random(insert_seed + 1, remove_seed + 1, 4));
    assert!(check_random(insert_seed + 2, remove_seed + 2, 4));
    assert!(check_random(insert_seed + 3, remove_seed + 3, 4));
    assert!(check_random(insert_seed + 0, remove_seed + 0, 5));
    assert!(check_random(insert_seed + 1, remove_seed + 1, 5));
    assert!(check_random(insert_seed + 2, remove_seed + 2, 5));
    assert!(check_random(insert_seed + 3, remove_seed + 3, 5));
    assert!(check_random(insert_seed, remove_seed, 10));
    assert!(check_random(insert_seed, remove_seed, 100));
    assert!(check_random(insert_seed, remove_seed, 1000));
    assert!(check_random(insert_seed, remove_seed, 10000));
    assert!(check_random(insert_seed, remove_seed, 100000));
    println!("OK")
}

## 5-2. Compile

In [ ]:
%%bash_
. ~/.cargo/env
rustc rs/tttree.rs -o rs/tttree

* Note: when you run `rustc` or other Rust commands in a terminal (SSH or Jupyter terminal), you need to execute the first line (`. ~/.cargo/env`)
* You may consider adding that line in your `~/.bash_profile`

## 5-3. Run

In [ ]:
%%bash_
rs/tttree

## 5-4. Ask Questions or Get Feedback

In [ ]:
%%hey problem_file=tttree.md answer_file=rs/tttree.rs

Problem:
{problem}

My Answer (between /** begin my answer */ and /** end my answer */):
{answer}

Give me a feedback to my answer.